# argentina.departamentos — Pruebas interactivas

Recorrido paso a paso del módulo `argentina.departamentos`.

Datos embebidos (no requiere internet). API inspirada en el paquete `us`.

**Nota:** por ahora el set incluye un subconjunto representativo de departamentos (no todos los del país).

## 1. Setup e imports

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")
print(f"total departamentos: {len(arg.departamentos.listar())}")

argentina v0.0.14
total departamentos: 13


## 2. Estructura del set

`Departamento` es un dataclass frozen con 4 campos.

In [2]:
# Primer departamento de la lista
arg.departamentos.listar()[0]

Departamento(codigo_departamento='02014', nombre='Comuna 1', provincia_codigo='02', provincia_nombre='Ciudad Autónoma de Buenos Aires')

In [3]:
# Campos individuales
d = arg.departamentos.lookup("06441")
print("codigo_departamento:", d.codigo_departamento)
print("nombre:             ", d.nombre)
print("provincia_codigo:   ", d.provincia_codigo)
print("provincia_nombre:   ", d.provincia_nombre)

codigo_departamento: 06441
nombre:              La Matanza
provincia_codigo:    06
provincia_nombre:    Buenos Aires


In [4]:
# frozen: no se puede mutar
try:
    d.nombre = "otro"
except Exception as e:
    print(type(e).__name__, '→', e)

FrozenInstanceError → cannot assign to field 'nombre'


## 3. Lookup por código de departamento

El código INDEC de departamento siempre identifica unívocamente un departamento.

In [5]:
arg.departamentos.lookup("06441")

Departamento(codigo_departamento='06441', nombre='La Matanza', provincia_codigo='06', provincia_nombre='Buenos Aires')

In [6]:
arg.departamentos.lookup("82084")

Departamento(codigo_departamento='82084', nombre='Rosario', provincia_codigo='82', provincia_nombre='Santa Fe')

In [7]:
# Los 4 "Capital" se distinguen por código
for codigo in ["14014", "54028", "66056", "90084"]:
    print(codigo, '→', arg.departamentos.lookup(codigo))

14014 → Departamento(codigo_departamento='14014', nombre='Capital', provincia_codigo='14', provincia_nombre='Córdoba')
54028 → Departamento(codigo_departamento='54028', nombre='Capital', provincia_codigo='54', provincia_nombre='Misiones')
66056 → Departamento(codigo_departamento='66056', nombre='Capital', provincia_codigo='66', provincia_nombre='Salta')
90084 → Departamento(codigo_departamento='90084', nombre='Capital', provincia_codigo='90', provincia_nombre='Tucumán')


## 4. Lookup por nombre

Sólo funciona cuando el nombre es **único** dentro del set. Es case-insensitive y tolera tildes.

In [8]:
# Nombre único: Rosario aparece sólo una vez
arg.departamentos.lookup("Rosario")

Departamento(codigo_departamento='82084', nombre='Rosario', provincia_codigo='82', provincia_nombre='Santa Fe')

In [9]:
# Case-insensitive
arg.departamentos.lookup("rosario")

Departamento(codigo_departamento='82084', nombre='Rosario', provincia_codigo='82', provincia_nombre='Santa Fe')

In [10]:
# Tolera tildes
print(arg.departamentos.lookup("General Pueyrredón"))
print(arg.departamentos.lookup("general pueyrredon"))

Departamento(codigo_departamento='06427', nombre='General Pueyrredón', provincia_codigo='06', provincia_nombre='Buenos Aires')
Departamento(codigo_departamento='06427', nombre='General Pueyrredón', provincia_codigo='06', provincia_nombre='Buenos Aires')


In [11]:
# Nombre ambiguo: "Capital" aparece 4 veces (Córdoba, Misiones, Salta, Tucumán)
# → devuelve None, hay que usar el código
arg.departamentos.lookup("Capital") is None

True

In [12]:
# "La Capital" (Santa Fe) sí es único
arg.departamentos.lookup("La Capital")

Departamento(codigo_departamento='82126', nombre='La Capital', provincia_codigo='82', provincia_nombre='Santa Fe')

## 5. Aliases

Atajos para nombres habituales sin tildes.

In [13]:
arg.departamentos.lookup("la matanza")

Departamento(codigo_departamento='06441', nombre='La Matanza', provincia_codigo='06', provincia_nombre='Buenos Aires')

In [14]:
arg.departamentos.lookup("rio cuarto")

Departamento(codigo_departamento='14126', nombre='Río Cuarto', provincia_codigo='14', provincia_nombre='Córdoba')

## 6. Filtrar por provincia

`por_provincia` acepta el nombre de la provincia (con o sin tildes) o su código INDEC.

In [15]:
# Por nombre
arg.departamentos.por_provincia("Buenos Aires")

(Departamento(codigo_departamento='06028', nombre='Avellaneda', provincia_codigo='06', provincia_nombre='Buenos Aires'),
 Departamento(codigo_departamento='06427', nombre='General Pueyrredón', provincia_codigo='06', provincia_nombre='Buenos Aires'),
 Departamento(codigo_departamento='06441', nombre='La Matanza', provincia_codigo='06', provincia_nombre='Buenos Aires'))

In [16]:
# Por código
arg.departamentos.por_provincia("06")

(Departamento(codigo_departamento='06028', nombre='Avellaneda', provincia_codigo='06', provincia_nombre='Buenos Aires'),
 Departamento(codigo_departamento='06427', nombre='General Pueyrredón', provincia_codigo='06', provincia_nombre='Buenos Aires'),
 Departamento(codigo_departamento='06441', nombre='La Matanza', provincia_codigo='06', provincia_nombre='Buenos Aires'))

In [17]:
# Sin tildes
arg.departamentos.por_provincia("cordoba")

(Departamento(codigo_departamento='14014', nombre='Capital', provincia_codigo='14', provincia_nombre='Córdoba'),
 Departamento(codigo_departamento='14126', nombre='Río Cuarto', provincia_codigo='14', provincia_nombre='Córdoba'))

In [18]:
# Provincia sin departamentos en el set → tupla vacía
arg.departamentos.por_provincia("Mendoza")

()

## 7. Casos borde

Valores inválidos o vacíos no rompen — devuelven `None` o tupla vacía.

In [19]:
print(arg.departamentos.lookup("99999"))           # código inexistente
print(arg.departamentos.lookup("Mar del Plata"))   # nombre no incluido
print(arg.departamentos.lookup(""))                 # string vacío
print(arg.departamentos.lookup(None))               # None
print(arg.departamentos.por_provincia(None))        # None → ()
print(arg.departamentos.por_provincia("Atlantis")) # provincia inexistente → ()

None
None
None
None
()
()


## 8. Iterar y agrupar

`listar()` devuelve una tupla inmutable.

In [20]:
for d in arg.departamentos.listar():
    print(f"{d.codigo_departamento}  {d.provincia_nombre:35s}  {d.nombre}")

02014  Ciudad Autónoma de Buenos Aires      Comuna 1
02021  Ciudad Autónoma de Buenos Aires      Comuna 2
06028  Buenos Aires                         Avellaneda
06427  Buenos Aires                         General Pueyrredón
06441  Buenos Aires                         La Matanza
14014  Córdoba                              Capital
14126  Córdoba                              Río Cuarto
22028  Chaco                                Comandante Fernández
54028  Misiones                             Capital
66056  Salta                                Capital
82084  Santa Fe                             Rosario
82126  Santa Fe                             La Capital
90084  Tucumán                              Capital


In [21]:
# Agrupar por provincia
from collections import defaultdict

por_prov = defaultdict(list)
for d in arg.departamentos.listar():
    por_prov[d.provincia_nombre].append(d.nombre)

for prov, nombres in sorted(por_prov.items()):
    print(f"{prov}: {', '.join(nombres)}")

Buenos Aires: Avellaneda, General Pueyrredón, La Matanza
Chaco: Comandante Fernández
Ciudad Autónoma de Buenos Aires: Comuna 1, Comuna 2
Córdoba: Capital, Río Cuarto
Misiones: Capital
Salta: Capital
Santa Fe: Rosario, La Capital
Tucumán: Capital


In [22]:
# Combinar con argentina.provincias para enriquecer
from argentina import provincias

for d in arg.departamentos.por_provincia("Santa Fe"):
    p = provincias.lookup(d.provincia_codigo)
    print(f"{d.nombre} — provincia {p.nombre} (capital: {p.capital}, región: {p.region})")

Rosario — provincia Santa Fe (capital: Santa Fe, región: Pampeana)
La Capital — provincia Santa Fe (capital: Santa Fe, región: Pampeana)


## 9. Tests automáticos

Los tests viven en `tests/test_departamentos.py`. Para correrlos:

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_departamentos.py -v
```

## Notas sueltas / TODOs

- Módulo sin dependencias externas (sólo stdlib). Datos embebidos en `src/argentina/data/departamentos.csv`.
- Por ahora el set es mínimo (13 departamentos representativos) — la idea es validar la API antes de expandir.
- Para agregar más departamentos: editar el CSV. La detección de nombres duplicados es automática (`_LOOKUP` se reconstruye al importar).
- Para agregar más aliases (ej. `mar del plata` → General Pueyrredón): editar `_ALIASES` en `departamentos.py`.